# Toy exercise - simple time-to-event
Trying to get transformer working for a simple survival prediction

In [1]:
import numpy as np
import pandas as pd
import random
# load et simulation function from simulate_population.py: 
from simulate_population import sim_population

### Simulate simple 1-step no conditional dependencies population and outcomes

In [57]:
horizon = 10
pop = sim_population(N=20000, step_forward=horizon, randomseed=42)
df_short = pop.history[0].iloc[0:10000,:].copy()
df_test = pop.history[0].iloc[10000:20000,:].copy()


In [69]:
events = ["a", "b", "c", "d", "e"]
# change time_a to censoring time or event time until "horizon" 
for e in events:
    df_short[f"time_{e}"]= np.minimum(horizon,df_short[f"time_{e}"])
    df_test[f"time_{e}"]= np.minimum(horizon,df_test[f"time_{e}"])
df_short.head(10)

,id,start,end,age_start,bmi,hyp,smoke,sex,eth,first_a,...,time_a,time_b,time_c,time_d,time_e,event_a,event_b,event_c,event_d,event_e
0,1,0.0,10,62.1,-1.8,0,0,1,0,5.199,...,5.199,0.059,3.133,10.000,10.0,1,1,1,0,0
1,2,0.0,10,43.0,-0.4,0,0,1,2,4.363,...,4.363,4.903,7.496,10.000,10.0,1,1,1,0,0
2,3,0.0,10,66.9,-1.7,0,0,0,0,NaN,...,10.000,1.925,0.860,10.000,10.0,0,1,1,0,0
3,4,0.0,10,57.7,2.0,1,0,0,2,0.100,...,0.100,0.382,0.095,10.000,10.0,1,1,1,0,0
4,5,0.0,10,23.4,0.2,0,0,1,0,4.573,...,4.573,10.000,10.000,10.000,10.0,1,0,0,0,0
5,6,0.0,10,73.6,0.1,0,0,0,0,1.077,...,1.077,0.044,0.344,10.000,10.0,1,1,1,0,0
6,7,0.0,10,61.4,0.2,0,0,1,1,3.199,...,3.199,10.000,10.000,2.601,10.0,1,0,0,1,0
7,8,0.0,10,62.8,0.6,0,1,1,1,0.313,...,0.313,0.460,8.824,10.000,10.0,1,1,1,0,0
8,9,0.0,10,25.3,1.0,1,0,1,0,5.106,...,5.106,2.404,10.000,10.000,10.0,1,1,0,0,0
9,10,0.0,10,43.7,-1.6,0,0,1,0,NaN,...,10.000,1.378,1.328,10.000,10.0,0,1,1,0,0


### CoxPH

In [167]:
import pandas as pd
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
import matplotlib.pyplot as plt

In [177]:
params = ["age_start","bmi","hyp","smoke","sex","eth"]
printcoef = False

In [183]:
for e in events:
    cph = CoxPHFitter()
    cph.fit(df_short[params + [f"time_{e}", f"event_{e}"]], duration_col=f"time_{e}", event_col=f"event_{e}")
    s =cph.summary[['coef', 'se(coef)', 'p']]
    if printcoef: 
        print(round(s,4))
    risk = cph.predict_partial_hazard(df_test[params])
    cindex = concordance_index(df_test[f"time_{e}"], -1*risk, df_test[f"event_{e}"])
    print("C-index for event", e, " =",np.round(cindex,3))

C-index for event a  = 0.708
C-index for event b  = 0.64
C-index for event c  = 0.642
C-index for event d  = 0.542
C-index for event e  = 0.547


In [122]:
surv = cph.predict_survival_function(df_test[params])
# survival at t = 9.512
# surv.loc[9.9512]
#plt.figure(figsize = (3,1))
#plt.scatter(risk,surv.loc[1.012])

In [ ]:
class SurvivalNN(nn.Module):
    def __init__(self, vocab_size, d_model=64):
        super().__init__()
        self.time_emb = nn.Linear(1, d_model)   # dt
        self.linear1 = nn.Linear(d_model, 2*d_model)    # age, hyp, smoke, sex, eth
        self.linear2 = nn.Linear(2*d_model, d_model)    # age, hyp, smoke, sex, eth
        self.out = nn.Linear(d_model, vocab_size)
    
    def forward(self, token_ids, dt, cov_time, cov_invar):
        x = self.encoder(x)
        
        logits = self.out(x)
        return logits